# 4.8 数值稳定性和模型初始化

深层网络的梯度是许多层局部导数的连乘。即使每一层只让梯度略微缩小或放大，跨越许多层后也可能产生梯度消失或梯度爆炸。初始化的目标不是猜中最终参数，而是让优化从一个信号与梯度都能稳定传播的起点开始。

原教材：[4.8 数值稳定性和模型初始化](https://zh.d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html)

## 学习目标

1. 从链式法则理解梯度消失与梯度爆炸；
2. 理解 sigmoid 饱和、深层矩阵连乘和有限精度带来的问题；
3. 解释为什么所有隐藏单元不能使用完全相同的初始化；
4. 推导 Xavier 初始化的方差条件，并了解 ReLU 常用的 Kaiming 初始化。

## 4.8.1 梯度消失与梯度爆炸

对 $L$ 层网络，反向传播包含雅可比矩阵的连乘：

$$\frac{\partial \mathbf h^{(L)}}{\partial \mathbf h^{(0)}}=\prod_{l=1}^{L}\frac{\partial \mathbf h^{(l)}}{\partial \mathbf h^{(l-1)}}.$$

若这些因子的典型尺度小于 1，乘积会指数级趋近 0，早期层几乎得不到学习信号；若大于 1，乘积会快速增大，导致更新剧烈、出现 `inf` 或 `nan`。这不是简单的“梯度小/大”，而是深度带来的乘法效应。

sigmoid 的导数最大只有 $1/4$，且输入绝对值较大时进入饱和区，导数接近 0。ReLU 正半轴导数为 1，缓解了饱和导致的梯度消失，但不合适的权重尺度仍会造成前向激活和反向梯度失控。

In [ ]:
from pathlib import Path  # 统一管理本地与 Google Drive 中的图片路径

import matplotlib.pyplot as plt  # 绘制不同深度下的梯度范数
import torch  # 提供张量计算、自动微分与初始化工具
from torch import nn  # 提供网络层和常见初始化方法

try:  # 尝试判断当前环境是否为 Google Colab
    from google.colab import drive  # 导入 Colab 的云盘挂载工具
    drive.mount('/content/drive')  # 挂载用户的“我的云端硬盘”
    project_root = Path('/content/drive/MyDrive/d2l_learning')  # 指向云盘中的项目根目录
except ImportError:  # 本地环境没有 google.colab 时进入此分支
    project_root = Path.cwd().parent if Path.cwd().name == 'Chapter_4' else Path.cwd()  # 推断本地项目根目录

figure_dir = project_root / 'Chapter_4' / 'images'  # 设置第 4 章图片保存目录
figure_dir.mkdir(parents=True, exist_ok=True)  # 创建图片目录且允许目录已存在
torch.manual_seed(42)  # 固定 CPU 随机种子以复现实验结果
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))  # 自动选择 Colab GPU 或本地 CPU
print('计算设备:', device)  # 输出当前实际使用的计算设备

In [ ]:
def input_gradient_norm(depth, activation, weight_scale):  # 定义测量深层网络输入梯度范数的函数
    X = torch.randn(64, 128, device=device, requires_grad=True)  # 创建需要梯度的小批量输入 [64,128]
    H = X  # 将当前隐藏状态初始化为网络输入
    for layer_index in range(depth):  # 依次构造并执行指定数量的隐藏层
        W = torch.randn(128, 128, device=device) * weight_scale  # 按指定尺度生成当前层权重
        H = H @ W  # 执行当前层的线性矩阵变换
        H = torch.sigmoid(H) if activation == 'sigmoid' else torch.relu(H)  # 根据名称选择 sigmoid 或 ReLU
    H.mean().backward()  # 对最终平均激活反向传播到原始输入
    return X.grad.norm().item()  # 返回输入梯度的 L2 范数

depths = [1, 2, 4, 8, 16, 32]  # 设置需要比较的网络深度
sigmoid_norms = [input_gradient_norm(depth, 'sigmoid', 1 / 128 ** 0.5) for depth in depths]  # 测量 sigmoid 网络各深度的梯度
relu_stable_norms = [input_gradient_norm(depth, 'relu', (2 / 128) ** 0.5) for depth in depths]  # 使用 Kaiming 尺度测量 ReLU 梯度
relu_large_norms = [input_gradient_norm(depth, 'relu', 0.3) for depth in depths]  # 使用过大权重尺度观察 ReLU 梯度爆炸
print('sigmoid 梯度范数:', sigmoid_norms)  # 输出 sigmoid 网络的梯度范数序列
print('ReLU 合理初始化梯度范数:', relu_stable_norms)  # 输出合理初始化下的 ReLU 梯度范数
print('ReLU 过大初始化梯度范数:', relu_large_norms)  # 输出过大初始化下的 ReLU 梯度范数

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4))  # 创建梯度范数比较图的画布
axis.semilogy(depths, sigmoid_norms, marker='o', label='sigmoid')  # 用对数纵轴绘制 sigmoid 梯度范数
axis.semilogy(depths, relu_stable_norms, marker='o', label='ReLU with Kaiming scale')  # 绘制合理初始化的 ReLU 梯度
axis.semilogy(depths, relu_large_norms, marker='o', label='ReLU with large weights')  # 绘制权重过大时的 ReLU 梯度
axis.set_xlabel('depth')  # 设置横轴为网络深度
axis.set_ylabel('input gradient norm')  # 设置纵轴为输入梯度范数
axis.grid(alpha=0.3)  # 添加半透明网格辅助读数
axis.legend()  # 显示三组实验对应的图例
plt.tight_layout()  # 自动调整画布边距
figure_path = figure_dir / '4.8_gradient_stability.png'  # 生成图片的完整保存路径
fig.savefig(figure_path, dpi=160, bbox_inches='tight')  # 保存高清图片并裁掉多余白边
print(f'图片已保存到：{figure_path}')  # 输出图片保存位置
plt.show()  # 在 Notebook 中显示梯度稳定性图

## 4.8.2 打破对称性

如果同一隐藏层的所有神经元拥有完全相同的权重，它们接收相同输入、产生相同输出，也会获得相同梯度。训练后这些神经元仍然相同，相当于只学习了一个特征。因此深层网络不能把全部权重初始化为同一个常数，尤其不能全部初始化为 0。

随机初始化的关键作用之一就是打破隐藏单元之间的置换对称性。偏置可以全零初始化，因为不同神经元的随机权重已经打破了对称性。

In [ ]:
X_demo = torch.tensor([[1.0, 2.0]], device=device)  # 创建用于对称性实验的单样本输入
zero_layer = nn.Linear(2, 3, bias=False).to(device)  # 创建包含三个隐藏单元的全连接层
nn.init.zeros_(zero_layer.weight)  # 将所有隐藏单元权重故意初始化为相同的零
zero_output = zero_layer(X_demo)  # 使用纯线性输出隔离并观察参数对称性问题
zero_output.sum().backward()  # 对输出总和反向传播以观察三个单元的梯度
print('相同初始化的隐藏输出:', zero_output.detach())  # 输出三个完全相同的激活值
print('相同初始化的权重梯度:', zero_layer.weight.grad)  # 输出无法打破对称性的相同梯度

## 4.8.3 Xavier 初始化

考虑无激活函数的线性层 $o_j=\sum_{i=1}^{n_{in}}w_{ij}x_i$。假设输入和权重独立、均值为 0，则

$$\operatorname{Var}(o_j)=n_{in}\operatorname{Var}(w_{ij})\operatorname{Var}(x_i).$$

为了让前向传播方差不变，希望 $n_{in}\operatorname{Var}(w)=1$；为了让反向传播梯度方差不变，希望 $n_{out}\operatorname{Var}(w)=1$。两者通常不能同时精确满足，Xavier 采用折中：

$$\operatorname{Var}(w)=\frac{2}{n_{in}+n_{out}}.$$

正态版标准差为 $\sqrt{2/(n_{in}+n_{out})}$；均匀版范围为 $[-\sqrt{6/(n_{in}+n_{out})},\sqrt{6/(n_{in}+n_{out})}]$。Xavier 更适合近似线性的 tanh；ReLU 会屏蔽约一半信号，常用方差 $2/n_{in}$ 的 Kaiming 初始化。

In [ ]:
xavier_layer = nn.Linear(128, 64).to(device)  # 创建输入宽度 128、输出宽度 64 的线性层
nn.init.xavier_normal_(xavier_layer.weight)  # 使用 Xavier 正态分布初始化权重
nn.init.zeros_(xavier_layer.bias)  # 将偏置初始化为零
expected_std = (2.0 / (128 + 64)) ** 0.5  # 根据 Xavier 公式计算理论标准差
observed_std = xavier_layer.weight.detach().std().item()  # 计算实际采样权重的标准差
print('Xavier 理论标准差:', expected_std)  # 输出理论标准差
print('Xavier 实际标准差:', observed_std)  # 输出有限样本下的实际标准差

## 4.8.4 实践中的稳定性工具

初始化只能改善训练起点，不能独自解决所有稳定性问题。现代网络还会结合非饱和激活、残差连接、归一化层、合理学习率、梯度裁剪和稳定的复合损失实现。遇到 `nan` 时应先检查数据范围、损失公式、学习率以及梯度范数，而不是盲目更换优化器。

## 4.8.5 小结与练习答案

- 深度使局部导数反复相乘，从而产生梯度消失或爆炸；
- 随机初始化既控制信号尺度，也负责打破隐藏单元对称性；
- Xavier 在前向与反向方差要求之间折中，Kaiming 针对 ReLU 修正方差；
- 初始化必须与激活函数、网络结构及优化超参数共同考虑。

**练习：为什么 ReLU 网络仍可能梯度消失？** 若大量预激活长期为负，ReLU 的局部导数为 0，这些路径不会传播梯度；此外，过小权重也会让矩阵连乘不断缩小。